# 01 Explore the data

What this notebook checks before any model is trained:

- how many essays there are, and how many are AI-generated
- how the essays spread over topics
- that label 1 really means AI (a sanity check)
- how long the essays are, and how many are too long for the transformers

Run `uv run python -m essaydetect.data` first, so the splits exist.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from essaydetect.data import LABEL_COL, SOURCE_COL, TEXT_COL, TOPIC_COL, load_split

# VS Code runs notebooks from the notebooks/ folder, so step up to the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# The three parts of the random split together are the whole cleaned dataset.
df = pd.concat(load_split("random", ROOT / "data/splits").values(), ignore_index=True)
print(f"{len(df):,} essays")
df.head()

## Class balance

In [ ]:
df[LABEL_COL].value_counts().rename({0: "student", 1: "AI"})

## Topics: how many essays, and what share of them is AI?

In [ ]:
by_topic = df.groupby(TOPIC_COL)[LABEL_COL].agg(n_essays="size", ai_share="mean")
by_topic.sort_values("n_essays", ascending=False)

## Sanity check: does label 1 really mean AI?

The `source` column says where each essay came from. Every source under label 0 should be
a student corpus, and every source under label 1 should be a language model. If you can see
which model wrote each AI essay, that is useful for the error analysis later.

In [ ]:
if SOURCE_COL in df.columns:
    display(df.groupby(LABEL_COL)[SOURCE_COL].value_counts().to_frame("n_essays"))
else:
    print("No source column in this file.")

## Essay length in words

If AI essays are clearly longer or shorter than student essays, length alone is a strong
signal. That is worth knowing, and worth mentioning in the report.

In [ ]:
df["n_words"] = df[TEXT_COL].str.split().str.len()
display(df.groupby(LABEL_COL)["n_words"].describe().rename(index={0: "student", 1: "AI"}))

fig, ax = plt.subplots(figsize=(7, 4))
for label, name in [(0, "student"), (1, "AI")]:
    ax.hist(df.loc[df[LABEL_COL] == label, "n_words"], bins=60, alpha=0.6, label=name)
ax.set_xlabel("Essay length (words)")
ax.set_ylabel("Number of essays")
ax.legend()
plt.show()

## How many essays are longer than the models can read?

Transformers read a limited number of *tokens*. A token is a word or a piece of a word,
so an essay has more tokens than words. This cell checks how many essays DistilBERT
(512 tokens) would cut off, and whether 1024 tokens is enough for ModernBERT.

It downloads the two tokenizers the first time. A warning about sequences longer than
512 is expected and harmless here: we only count tokens, we do not run the model.

In [ ]:
from transformers import AutoTokenizer

sample = df.sample(2000, random_state=42)
for model_name, limit in [("distilbert-base-uncased", 512), ("answerdotai/ModernBERT-base", 1024)]:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    n_tokens = sample[TEXT_COL].map(lambda text: len(tokenizer(text)["input_ids"]))
    print(
        f"{model_name}: median {n_tokens.median():.0f} tokens, "
        f"{(n_tokens > limit).mean():.1%} of essays longer than {limit}"
    )

## Read a few essays

In [ ]:
for label, name in [(0, "STUDENT"), (1, "AI")]:
    essay = df[df[LABEL_COL] == label].sample(1, random_state=1).iloc[0]
    print(f"--- {name} essay, topic: {essay[TOPIC_COL]} ---")
    print(essay[TEXT_COL][:1500], "\n")

## Notes for the report

Fill these in as you go:

- Essays after cleaning:
- Share of AI essays:
- Test topics in the topic split (see `results/splits/split_summary.json`):
- Length difference between student and AI essays:
- Share of essays cut off at 512 tokens: